### Aim
Implement Artificial Neural Network training process in Python by using Forward Propagation, Back Propagation.

### Step 1: Import Required Packages

In [1]:
import numpy as np
import pandas as pd

### Step 2: Define ANN Model Class

In [2]:
class NeuralNetwork:
  def __init__(self, input_size, hidden_size, output_size):
    # Initialize weights and biases
    self.W1 = np.random.randn(input_size, hidden_size)
    self.b1 = np.zeros((1, hidden_size))
    self.W2 = np.random.randn(hidden_size, output_size)
    self.b2 = np.zeros((1, output_size))
      
  def sigmoid(self, x):
    return 1 / (1 + np.exp(-x))
  
  def sigmoid_derivative(self, x):
    return x * (1 - x)
  
  def forward(self, X):
    # Forward propagation
    self.z1 = np.dot(X, self.W1) + self.b1
    self.a1 = self.sigmoid(self.z1)
    self.z2 = np.dot(self.a1, self.W2) + self.b2
    self.a2 = self.sigmoid(self.z2)
    return self.a2
  
  def backward(self, X, y, output, learning_rate):
    # Back propagation
    # Output layer error
    output_error = y - output
    output_delta = output_error * self.sigmoid_derivative(output)
    
    # Hidden layer error
    hidden_error = np.dot(output_delta, self.W2.T)
    hidden_delta = hidden_error * self.sigmoid_derivative(self.a1)
    
    # Update weights and biases
    self.W2 += np.dot(self.a1.T, output_delta) * learning_rate
    self.b2 += np.sum(output_delta, axis=0, keepdims=True) * learning_rate
    self.W1 += np.dot(X.T, hidden_delta) * learning_rate
    self.b1 += np.sum(hidden_delta, axis=0, keepdims=True) * learning_rate
  
  def train(self, X, y, epochs, learning_rate):
    for epoch in range(epochs):
      output = self.forward(X)
      self.backward(X, y, output, learning_rate)
      if epoch % 100 == 0:
        loss = np.mean(np.square(y - output))
        print(f'Epoch {epoch}, Loss: {loss}')

### Step 3: Generate synthetic data: classify points inside a circle
* **Circle** : x1 ** 2 + x2 ** 2  =  1.5 ** 2
* **Points** : Random points in range [-2,2]
* **Inference** : If point (x1_i, x2_i) inside circle then y_i = 1 otherwise y_i = 0

In [3]:
np.random.seed(42)
n_samples = 200
X = np.random.uniform(-2, 2, (n_samples, 2))                         # Random points from [-2,2]
y = (X[:, 0]**2 + X[:, 1]**2 < 1.5**2).astype(int).reshape(-1, 1)    # Points inside circle of radius 1.5 : 1, others : 0

# Split into train and test
train_size = 150
X_train = X[:train_size]
y_train = y[:train_size]
X_test = X[train_size:]
y_test = y[train_size:]

In [9]:
X[:5] , y[:5]

(array([[-0.50183952,  1.80285723],
        [ 0.92797577,  0.39463394],
        [-1.37592544, -1.37602192],
        [-1.76766555,  1.46470458],
        [ 0.40446005,  0.83229031]]),
 array([[0],
        [1],
        [0],
        [0],
        [1]]))

### Step 4: Create and Train ANN Model

In [5]:
# Neural Model object creation
nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1)
nn.train(X_train, y_train, epochs=10000, learning_rate=0.1)

Epoch 0, Loss: 0.25303191677635684
Epoch 100, Loss: 0.07661214615324442
Epoch 200, Loss: 0.04653985219685897
Epoch 300, Loss: 0.032211921416807204
Epoch 400, Loss: 0.02591880500811826
Epoch 500, Loss: 0.02232332079488024
Epoch 600, Loss: 0.0199419694564374
Epoch 700, Loss: 0.018218769617315354
Epoch 800, Loss: 0.016895022379004043
Epoch 900, Loss: 0.01583346038045117
Epoch 1000, Loss: 0.014954362057908123
Epoch 1100, Loss: 0.014208193906377887
Epoch 1200, Loss: 0.013562512365562458
Epoch 1300, Loss: 0.012995145290862377
Epoch 1400, Loss: 0.01249039788772419
Epoch 1500, Loss: 0.012036823314633157
Epoch 1600, Loss: 0.011625852213663052
Epoch 1700, Loss: 0.01125091810049625
Epoch 1800, Loss: 0.010906881684796274
Epoch 1900, Loss: 0.01058964223013444
Epoch 2000, Loss: 0.01029586966348989
Epoch 2100, Loss: 0.010022816642356614
Epoch 2200, Loss: 0.009768184624753292
Epoch 2300, Loss: 0.009530026965544313
Epoch 2400, Loss: 0.009306677701040152
Epoch 2500, Loss: 0.009096698341172924
Epoch 2600

### Step 5: Testing of ANN Model

In [6]:
# Test on unseen data
predictions = nn.forward(X_test)
binary_preds = (predictions > 0.5).astype(int)
inside_circle = np.where(binary_preds == 1, "Yes", "No")
accuracy = np.mean(binary_preds == y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")
print("\nSample predictions:")
preds_df = pd.DataFrame({
    "Input (X1)": X_test[:, 0],
    "Input (X2)": X_test[:, 1],
    "Predicted": binary_preds[:, 0],
    "Actual": y_test[:, 0],
    "Inside Circle": inside_circle[:, 0]
})
preds_df.head()

Test Accuracy: 88.00%

Sample predictions:


,Input (X1),Input (X2),Predicted,Actual,Inside Circle
0,-1.793273,0.125419,0,0,No
1,0.162540,0.549720,1,1,Yes
2,0.904365,1.903408,0,0,No
3,0.065201,-0.708174,1,1,Yes
4,1.180745,-0.916671,1,1,Yes
